In [4]:
import pandas as pd
from sqlalchemy import create_engine


db_config = {'user': 'practicum_student',         # nombre de usuario
             'pwd': 's65BlTKV3faNIGhmvJVzOqhs', # contraseña
             'host': 'rc1b-wcoijxj3yxfsf3fs.mdb.yandexcloud.net',
             'port': 6432,              # puerto de conexión
             'db': 'data-analyst-final-project-db'}          # nombre de la base de datos

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(db_config['user'],
                                                                     db_config['pwd'],
                                                                       db_config['host'],
                                                                       db_config['port'],
                                                                       db_config['db'])

engine = create_engine(connection_string, connect_args={'sslmode':'require'})


In [ ]:

#1 número de libros publicados desde 01-ene-2000
query = """
SELECT COUNT(*)
FROM books
WHERE publication_date > '2000-01-01'
"""
pd.read_sql(query, engine)




,count
0,819


In [ ]:
#2 número de reseñas de usuarios 
query = """
SELECT COUNT(*)
FROM reviews
"""
pd.read_sql(query, engine)


,count
0,2793


In [50]:
#3 y calificación promedio / libro
query = """
SELECT books.title, 
        AVG(ratings.rating) AS avg_rating
FROM ratings    
INNER JOIN books 
ON ratings.book_id = books.book_id   
GROUP BY books.title
ORDER BY avg_rating DESC;
"""
pd.read_sql(query, engine)

,title,avg_rating
0,Captivating: Unveiling the Mystery of a Woman'...,5.00
1,Evening Class,5.00
2,In the Hand of the Goddess (Song of the Liones...,5.00
3,The Big Bad Wolf (Alex Cross #9),5.00
4,A Dirty Job (Grim Reaper #1),5.00
...,...,...
994,The World Is Flat: A Brief History of the Twen...,2.25
995,His Excellency: George Washington,2.00
996,Junky,2.00
997,Drowning Ruth,2.00


In [49]:
#4 Editoriales con mayor número de libros publicados de más de 50 pg.
query = """
SELECT publisher, COUNT(book_id) AS num_books
FROM publishers    
INNER JOIN books 
ON publishers.publisher_id = books.publisher_id   
WHERE num_pages > 50
GROUP BY publisher
ORDER BY num_books DESC;

"""
pd.read_sql(query, engine)

,publisher,num_books
0,Penguin Books,42
1,Vintage,31
2,Grand Central Publishing,25
3,Penguin Classics,24
4,Ballantine Books,19
...,...,...
329,Turtleback,1
330,Atheneum Books for Young Readers: Richard Jack...,1
331,Penguin Signet,1
332,Victor Gollancz,1


In [53]:
#5 Autor con más alta calificación promedio del libro (libros a partir de 50 calificaciones)

query = """
SELECT a.author, AVG(r.rating) AS avg_rating 
FROM authors a   
INNER JOIN books b
ON a.author_id = b.author_id 
INNER JOIN ratings r
ON b.book_id = r.book_id    
WHERE b.book_id IN(
        SELECT book_id
        FROM ratings
        GROUP BY book_id
        HAVING COUNT(*) >= 50
)
GROUP BY a.author
ORDER BY avg_rating DESC
LIMIT 1
"""
pd.read_sql(query, engine)

,author,avg_rating
0,J.K. Rowling/Mary GrandPré,4.287097


In [10]:
#6 número promedio de reseñas, entre usuarios con más de 50 libros calificados
query = """
SELECT AVG(total_reviews) AS avg_reviews              -- promedio final de reseñas
FROM (
    SELECT username,                                 -- usuario
           COUNT(review_id) AS total_reviews          -- número de reseñas por usuario
    FROM reviews
    WHERE username IN (
        SELECT username                              -- usuarios con muchas calificaciones
        FROM ratings
        GROUP BY username
        HAVING COUNT(rating_id) > 50                 -- más de 50 libros calificados
    )
    GROUP BY username
) sub;
"""

 “No fue posible obtener resultados ya que no existen coincidencias entre los usuarios de las tablas ratings y reviews, por lo que ningún usuario cumple ambas condiciones.”